In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()))))
sys.path.append(os.path.abspath(os.path.join("../..", "src")))

In [2]:
import numpy as np
import torch

from data.data_generator import ExampleScalarFieldThreeDimensions, ExampleVectorialFieldThreeDimensions

In [3]:
import tensopsciml as to

from tensopsciml import TensOps
from tensopsciml.kernels import Derivative
from tensopsciml.calculus import ConnectOps
from tutorials.utils.utilities import mae

# cov_operators validation

This notebook validates `cov_derivative`, `cov_divergence`, and `cov_curl` (from `tensopsciml.calculus.cov_operators`) against their flat counterparts `partial_derivative`, `divergence`, and `curl` (from `tensopsciml.calculus.operators`), on the same Euclidean metric and fields used in `operators_validation.ipynb`.

The covariant operators take an affine connection instead of a bare metric. For a **constant** Euclidean metric the Levi-Civita connection's Christoffel symbols vanish identically, so every covariant operator collapses to its flat counterpart bit-for-bit -- that exact collapse is what each cell below checks.

In [4]:
N_data = 10
dx = dy = dz = 0.1
indexing='xy'

D = Derivative(dx, dy, dz, indexing=indexing).three_dimensions()

In [5]:
A_ = ExampleVectorialFieldThreeDimensions(N_data, 
    Fx_char='sin(x)**2 + cos(y)', Fy_char='sin(x)', Fz_char='sin(z)',
    dx=dx, dy=dy, dz=dz, indexing=indexing).Func()

phi_ = ExampleScalarFieldThreeDimensions(N_data, f_char='sin(x) + cos(y)', dx=dx, dy=dy, dz=dz, indexing=indexing).func()

phi = TensOps(phi_, domain_dim=3, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None)
A = TensOps(A_, domain_dim=3, codomain_dim=[3], order=[1, 0], contravariant_indices=[0], covariant_indices=None)

### Metric

`cov_divergence(A, connection, D)` differentiates once (via `cov_derivative`) before contracting, so the connection it needs lives on `A`'s own grid (`G0`). Only the base (`G0`) metric/inverse/volume-form are built here; whenever a cell below needs one at a coarser grid (`cov_curl`'s `lc_symbol`/`volume_form`, or a second connection for the rank-2 `cov_divergence` example), it restricts `metric_0`/`volume_form_0` with `to.m()` inline, right where it's needed.

In [6]:
G0 = list(A.values.shape[-3:])

metric_0_ = torch.eye(3, dtype=A_.dtype).reshape(1, 3, 3, 1, 1, 1).expand(N_data, 3, 3, *G0).clone()
metric_0 = to.MetrOps(metric_0_, domain_dim=3, codomain_dim=[3, 3])
inv_metric_0 = to.inverse(metric_0)
volume_form_0 = to.VolFormOps(metric_0)

### Levi-Civita tensor

`cov_curl` needs the same raw (unscaled) Levi-Civita symbol as `curl`, wrapped as a fully contravariant tensor `eps^{ijk}` -- it divides by the volume form itself. Only the base (`G0`) version is built here; `to.m(e_0)` gives the one node smaller version `cov_curl` needs.

In [7]:
e_0 = TensOps(to.levi_civita_symbol(N_data, 3, G0), domain_dim=3, codomain_dim=[3, 3, 3], order=[3, 0], contravariant_indices=[0, 1, 2], covariant_indices=None)

### Connection

The Levi-Civita connection of a metric, `Gamma^k_{ij} = (1/2) g^{kl}(\partial_i g_{jl} + d_j g_{il} - d_l g_{ij})`. It is built from `metric` differentiated by `D`, so it lives on `metric`'s own grid but has coefficients evaluated at the grid one node smaller (matching `partial_derivative(metric, D)`).

In [8]:
connection_0 = ConnectOps.levi_civita(metric_0, D)

## Christoffel symbols vanish for a constant Euclidean metric

$$ \Gamma^k_{ij} = 0 $$

In [9]:
left = connection_0.values
right = torch.zeros_like(connection_0.values)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


## cov_derivative reduces to partial_derivative (scalar field)

$$ \nabla_i \phi = \partial_i \phi $$

In [10]:
left = to.cov_derivative(phi, connection_0, D)
right = to.partial_derivative(phi, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


## cov_derivative reduces to partial_derivative (vector field)

$$ \nabla_j A^i = d_j A^i $$

In [11]:
left = to.cov_derivative(A, connection_0, D)
right = to.partial_derivative(A, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


## cov_divergence reduces to divergence

$$ \nabla_i A^i = \frac{1}{\sqrt{g}} \partial_i(\sqrt{g}\, A^i) $$

In [12]:
left = to.cov_divergence(A, connection_0, D)
right = to.divergence(A, volume_form_0, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


## cov_curl reduces to curl

`cov_curl` does not lower any index itself, so it is called on `lower_index(metric, A)`, exactly as its docstring prescribes for reproducing the flat `curl`.

$$ (\text{cov\_curl})^i = E^{ijk} \nabla_j A_k = (\text{curl}\, A)^i $$

In [13]:
e_1 = to.m(e_0)
volume_form_1 = to.m(volume_form_0)

left = to.cov_curl(to.lower_index(metric_0, A), connection_0, e_1, volume_form_1, D, index=0)
right = to.curl(A, metric_0, e_1, volume_form_1, D)

print(f"MAE = {mae(left, right)}")

MAE = 0.0


## cov_divergence generalizes beyond vectors

Unlike `divergence` (vector-only), `cov_divergence` accepts any tensor and an `index` selecting which contravariant slot to contract with the derivative. Applied to $\nabla A$ (the rank-2 tensor `partial_derivative(A, D)`), contracting its `index=0` (A's own) contravariant slot against the derivative reproduces $\nabla(\nabla \cdot A)$, i.e. the (lowered) gradient of the ordinary divergence.

$$ \nabla_k(\nabla_i A^i) = \nabla_k \left(\frac{1}{\sqrt{g}} \partial_i(\sqrt{g}\, A^i)\right) $$

In [14]:
metric_1 = to.m(metric_0)
connection_1 = ConnectOps.levi_civita(metric_1, D)
inv_metric_2 = to.inverse(to.m(metric_1))

gradA = to.partial_derivative(A, D)
left = to.cov_divergence(gradA, connection_1, D, index=0)
right = to.lower_index(to.m(metric_1), to.gradient(to.divergence(A, volume_form_0, D), inv_metric_2, D))

print(f"MAE = {mae(left, right)}")

MAE = 3.1426579880644567e-06
